<a href="https://colab.research.google.com/github/Kaveesha-Vihanga/DS_Project/blob/component-3/Component_3_Implementation_AOS_triplets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm import tqdm

# Enable tqdm for pandas
tqdm.pandas()

# Load your dataset
df = pd.read_csv('/content/drive/MyDrive/DSGP Component 3/sri_lanka_hotel_reviews_english.csv', sep=',')
print(f"Loaded {len(df)} reviews")

# Load the AOS triplet model
model_name = "tafseer-nayeem/aspect-opinion-sentiment_AOS-triplet"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Move to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# Instruction prompt for the model (as specified in the model card)
instruction = (
    "Definition: The output will be the aspects (both implicit and explicit), "
    "the corresponding opinion/describing terms, and the sentiment polarity (positive, negative, neutral). "
    "Use the format aspect:opinion:sentiment. \n\n"
    "Example 1-\n"
    "input: Free Wi-Fi throughout the hotel\n"
    "output: wi-fi:free:positive\n\n"
    "Now complete the following example-\ninput: "
)

def extract_aspect_triplets(review_text):
    """Extract aspect-opinion-sentiment triplets from a review"""
    if not isinstance(review_text, str) or len(review_text.strip()) == 0:
        return []

    try:
        # Prepare input with instruction
        input_text = instruction + review_text + " \noutput:"

        # Tokenize
        inputs = tokenizer([input_text], return_tensors="pt",
                          truncation=True, max_length=512)
        inputs = {k: v.to(device) for k, v in inputs.items()}

        # Generate
        with torch.no_grad():
            outputs = model.generate(
                inputs['input_ids'],
                max_new_tokens=250,
                num_beams=4,
                early_stopping=True
            )

        # Decode
        decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

        # Parse the output (format: "aspect:opinion:sentiment, aspect:opinion:sentiment")
        triplets = []
        for triplet_str in decoded.split(','):
            parts = triplet_str.strip().split(':')
            if len(parts) == 3:
                aspect, opinion, sentiment = parts
                triplets.append({
                    'aspect': aspect.strip(),
                    'opinion': opinion.strip(),
                    'sentiment': sentiment.strip().lower()
                })

        return triplets

    except Exception as e:
        print(f"Error processing: {review_text[:50]}... Error: {e}")
        return []

# Apply to your dataset (this will take some time for 16k reviews)
print("Extracting aspect triplets...")
df['aspect_triplets'] = df['review_clean'].progress_apply(extract_aspect_triplets)

# Expand triplets into separate rows for analysis
expanded_rows = []
for idx, row in df.iterrows():
    for triplet in row['aspect_triplets']:
        expanded_rows.append({
            'user_name': row['User Name'],
            'address': row['Address'],
            'rating': row['Rating'],
            'review': row['review_clean'][:100] + '...',  # Truncate for display
            'aspect': triplet['aspect'],
            'opinion': triplet['opinion'],
            'sentiment': triplet['sentiment']
        })

# Create expanded dataframe
aspect_df = pd.DataFrame(expanded_rows)
print(f"Extracted {len(aspect_df)} aspect-opinion pairs from {len(df)} reviews")

# Save results
aspect_df.to_csv('/content/drive/MyDrive/DSGP Component 3/hotel_aspect_sentiments.csv', index=False)
df.to_csv('/content/drive/MyDrive/DSGP Component 3/reviews_with_triplets.csv', index=False)

Loaded 16252 reviews


Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Extracting aspect triplets...


100%|██████████| 16252/16252 [2:22:21<00:00,  1.90it/s]


Extracted 19282 aspect-opinion pairs from 16252 reviews
